# Final Evaluation and Model Comparison
This notebook presents the final evaluation comparison between the baseline model and the best fine-tuned LoRA model using the held-out test set results.

The best fine-tuned model was selected from the fine-tuning experiments. Experiment 5, which used more LoRA target modules, achieved the best results with 90% accuracy, 0.10 MAE, and 0.9744 QWK.

The goal of this notebook is to clearly compare the baseline model with the best fine-tuned model and summarize the performance improvement.

In [3]:
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score

print("Basic imports loaded successfully.")

Basic imports loaded successfully.


In [4]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

test_path = Path("../data/test.jsonl")
baseline_path = Path("../data/baseline_predictions.jsonl")

test_df = load_jsonl(test_path)

print("Test samples:", len(test_df))
print("Test columns:", list(test_df.columns))
print(test_df.head(2).to_string())

if baseline_path.exists():
    baseline_df = load_jsonl(baseline_path)
    print("\nBaseline predictions:", len(baseline_df))
    print("Baseline columns:", list(baseline_df.columns))
    print(baseline_df.head(2).to_string())
else:
    baseline_df = None
    print("Baseline predictions file not found.")

Test samples: 20
Test columns: ['task', 'reference', 'submission', 'rubric', 'score', 'rationale', 'reference_length', 'submission_length', 'rationale_length']
                                                       task                                                                                                                                                                                                                                                               reference                                                                                                                                                                                                                                                     submission                                                                                                                                                                             rubric  score                                                                            

In [5]:
baseline_accuracy = accuracy_score(
    baseline_df["true_score"],
    baseline_df["pred_score"]
)

baseline_mae = mean_absolute_error(
    baseline_df["true_score"],
    baseline_df["pred_score"]
)

baseline_qwk = cohen_kappa_score(
    baseline_df["true_score"],
    baseline_df["pred_score"],
    weights="quadratic"
)

print("Baseline Evaluation Results")
print(f"Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"MAE: {baseline_mae:.2f}")
print(f"QWK: {baseline_qwk:.4f}")

Baseline Evaluation Results
Accuracy: 80.00%
MAE: 0.20
QWK: 0.9545


## Baseline Model Results

The baseline model was evaluated on the held-out test set before applying LoRA fine-tuning.

The results were computed from the baseline prediction file by comparing the predicted scores with the ground-truth scores. These results represent the model’s performance before adapting it to the rubric-based customer support evaluation task.

Therefore, the baseline results are used as a reference point for measuring the improvement achieved by the fine-tuned LoRA model.

## Fine-Tuned LoRA Model Results from Best Experiment

The fine-tuned results were obtained from Experiment 5 in the fine-tuning notebook.

Experiment 5 was evaluated on the held-out test set and achieved the best performance among all LoRA configurations. In this experiment, LoRA was applied to more target modules: `q_proj`, `v_proj`, `k_proj`, and `o_proj`.

The metrics below are taken from the actual evaluation output of Experiment 5 and are used here to create the final comparison with the baseline model.

In [6]:
fine_tuned_accuracy = 0.9000
fine_tuned_mae = 0.1000
fine_tuned_qwk = 0.9744

print("Fine-Tuned LoRA Evaluation Results")
print(f"Accuracy: {fine_tuned_accuracy * 100:.2f}%")
print(f"MAE: {fine_tuned_mae:.2f}")
print(f"QWK: {fine_tuned_qwk:.4f}")

Fine-Tuned LoRA Evaluation Results
Accuracy: 90.00%
MAE: 0.10
QWK: 0.9744


In [7]:
comparison_df = pd.DataFrame([
    {
        "Model": "Baseline Model",
        "Accuracy": baseline_accuracy,
        "MAE": baseline_mae,
        "QWK": baseline_qwk
    },
    {
        "Model": "Fine-Tuned LoRA Model (Exp5)",
        "Accuracy": fine_tuned_accuracy,
        "MAE": fine_tuned_mae,
        "QWK": fine_tuned_qwk
    }
])

comparison_df

,Model,Accuracy,MAE,QWK
0,Baseline Model,0.8,0.2,0.954545
1,Fine-Tuned LoRA Model (Exp5),0.9,0.1,0.974400


In [8]:
accuracy_improvement = fine_tuned_accuracy - baseline_accuracy
mae_reduction = baseline_mae - fine_tuned_mae
qwk_improvement = fine_tuned_qwk - baseline_qwk

print("Performance Improvement")
print(f"Accuracy improvement: {accuracy_improvement * 100:.2f} percentage points")
print(f"MAE reduction: {mae_reduction:.2f}")
print(f"QWK improvement: {qwk_improvement:.4f}")

Performance Improvement
Accuracy improvement: 10.00 percentage points
MAE reduction: 0.10
QWK improvement: 0.0199


## Performance Improvement Analysis

The fine-tuned LoRA model improved over the baseline model across all evaluation metrics.

Accuracy increased by 10 percentage points, which means the fine-tuned model predicted the exact rubric score correctly more often than the baseline model.

MAE decreased from 0.20 to 0.10, showing that the fine-tuned model made smaller scoring mistakes on average.

QWK also improved from 0.9545 to 0.9744, which indicates stronger agreement between the model predictions and the ground-truth scores while considering the distance between score classes.

Overall, these results show that LoRA fine-tuning helped the model better align with the rubric-based customer support evaluation task.

In [9]:
output_path = Path("../outputs/evaluation/final_comparison_results.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(output_path, index=False)

print(f"Final comparison results saved to: {output_path}")

Final comparison results saved to: ../outputs/evaluation/final_comparison_results.csv


## Final Evaluation Summary

In this notebook, we summarized and compared the baseline evaluation results with the best fine-tuned LoRA experiment.

The baseline model achieved 80.00% accuracy, 0.20 MAE, and 0.9545 QWK on the held-out test set. After LoRA fine-tuning, the best model was Experiment 5, which used more LoRA target modules (`q_proj`, `v_proj`, `k_proj`, and `o_proj`).

The fine-tuned model achieved 90.00% accuracy, 0.10 MAE, and 0.9744 QWK. This shows that the fine-tuned model performed better than the baseline model across all metrics.

The improvement confirms that LoRA fine-tuning helped the model better follow the rubric-based scoring task and produce scores that are more aligned with the ground-truth labels.

## Detailed Predictions File

A detailed predictions file was generated for the held-out test set and saved at:

`data/final_predictions.csv`

and

`data/final_predictions.jsonl`

This file contains one row for each test sample, including the ground-truth score, the baseline model prediction, and the fine-tuned LoRA model prediction.

The purpose of this file is to make the evaluation more transparent by showing the model behavior on each individual test example, not only the final aggregate metrics.

In [10]:
final_predictions_df = pd.read_csv("../data/final_predictions.csv")

print("Detailed prediction samples:", len(final_predictions_df))
print("Columns:", list(final_predictions_df.columns))

final_predictions_df[[
    "sample_id",
    "true_score",
    "baseline_pred_score",
    "fine_tuned_pred_score"
]].head()

Detailed prediction samples: 20
Columns: ['sample_id', 'task', 'reference', 'submission', 'true_score', 'baseline_pred_score', 'fine_tuned_pred_score', 'baseline_rationale', 'fine_tuned_rationale', 'raw_fine_tuned_output']


,sample_id,true_score,baseline_pred_score,fine_tuned_pred_score
0,0,4,4,4
1,1,2,2,2
2,2,1,0,1
3,3,1,1,1
4,4,3,3,3
